In [4]:
import pandas as pd
import re
from underthesea import word_tokenize

# 1. Đọc dữ liệu THÔ vừa cào về
raw_path = r'D:\NNLT_PYTHON\GroupPython\d-o-n-nguy-c-kh-ch-h-ng-r-i-b-th-ng-qua-c-c-t-n-hi-u-b-t-m-n-trong-v-n-b-n-nh-gi-\project\data\raw\Shopee_Reviews.csv'
df = pd.read_csv(raw_path)

# 2. Xóa các dòng bị thiếu nội dung (NaN)
df = df.dropna(subset=['review_text'])

# 3. Lọc bỏ đánh giá 3 sao và Gán nhãn (1-2 sao = 1: Rủi ro | 4-5 sao = 0: An toàn)
df = df[df['rating'] != 3].copy()
df['risk_label'] = df['rating'].apply(lambda x: 1 if x <= 2 else 0)

# 4. HÀM LÀM SẠCH VĂN BẢN CHÍNH
def clean_text(text):
    text = str(text).lower() # Chuyển thành chữ thường
    text = re.sub(r'[^\w\s]', ' ', text) # Xóa dấu câu, icon, ký tự đặc biệt bằng Regex
    text = re.sub(r'\s+', ' ', text).strip() # Xóa khoảng trắng thừa
    text = word_tokenize(text, format="text") # Tách từ tiếng Việt (VD: "hài lòng" -> "hài_lòng")
    return text

# Áp dụng hàm làm sạch cho toàn bộ cột review_text để tạo ra cột mới tên là clean_text
print("Đang tiến hành làm sạch văn bản...")
df['clean_text'] = df['review_text'].apply(clean_text)

# 5. Xóa các dòng mà sau khi làm sạch bị rỗng (do khách chỉ gõ mỗi icon)
df = df[df['clean_text'].str.strip() != '']

# 6. Lưu file SẠCH vào thư mục processed để file vẽ biểu đồ của Khang có cái mà đọc
processed_path = r'D:\NNLT_PYTHON\GroupPython\d-o-n-nguy-c-kh-ch-h-ng-r-i-b-th-ng-qua-c-c-t-n-hi-u-b-t-m-n-trong-v-n-b-n-nh-gi-\project\data\processed\cleaned_shopee_reviews.csv'
df.to_csv(processed_path, index=False, encoding='utf-8-sig')

print(f"Đã làm sạch xong! Dữ liệu lưu tại: {processed_path}")

Đang tiến hành làm sạch văn bản...
Đã làm sạch xong! Dữ liệu lưu tại: D:\NNLT_PYTHON\GroupPython\d-o-n-nguy-c-kh-ch-h-ng-r-i-b-th-ng-qua-c-c-t-n-hi-u-b-t-m-n-trong-v-n-b-n-nh-gi-\project\data\processed\cleaned_shopee_reviews.csv


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

print("1. Đang nạp dữ liệu sạch...")
file_path = r'D:\NNLT_PYTHON\GroupPython\d-o-n-nguy-c-kh-ch-h-ng-r-i-b-th-ng-qua-c-c-t-n-hi-u-b-t-m-n-trong-v-n-b-n-nh-gi-\project\data\processed\cleaned_shopee_reviews.csv'

# ====================================================================
# SỬA Ở ĐÂY: Thêm on_bad_lines='skip' và engine='python' để vượt qua dòng lỗi
# ====================================================================
df = pd.read_csv(file_path, on_bad_lines='skip', engine='python')

# Đảm bảo không có dòng rỗng
df.dropna(subset=['clean_text', 'risk_label'], inplace=True)

print("2. Đang phân chia dữ liệu và chuyển Chữ thành Số (TF-IDF)...")
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['clean_text'], df['risk_label'], test_size=0.2, random_state=42, stratify=df['risk_label']
)

# Bổ sung ngram_range=(1, 2) khớp với nội dung báo cáo Word
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train = tfidf.fit_transform(X_train_raw)
X_test = tfidf.transform(X_test_raw)

print("3. Đang huấn luyện Logistic Regression (Baseline)...")
model = LogisticRegression(class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

# Dự đoán
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print("\n" + "="*50)
print(" 🎯 BÁO CÁO KẾT QUẢ MÔ HÌNH BASELINE ")
print("="*50)
print(classification_report(y_test, y_pred, target_names=['0 (An toàn)', '1 (Nguy cơ)']))

print("\n--- CÁC CHỈ SỐ BỔ SUNG ---")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba)*100:.2f}%\n")
print("Ma trận nhầm lẫn (Confusion Matrix):")
cm_df = pd.DataFrame(confusion_matrix(y_test, y_pred), 
                     index=['Thực tế: 0', 'Thực tế: 1'], 
                     columns=['Đoán: 0', 'Đoán: 1'])
print(cm_df)

1. Đang nạp dữ liệu sạch...
2. Đang phân chia dữ liệu và chuyển Chữ thành Số (TF-IDF)...
3. Đang huấn luyện Logistic Regression (Baseline)...

 🎯 BÁO CÁO KẾT QUẢ MÔ HÌNH BASELINE 
              precision    recall  f1-score   support

 0 (An toàn)       0.97      0.84      0.90       463
 1 (Nguy cơ)       0.80      0.96      0.87       302

    accuracy                           0.89       765
   macro avg       0.89      0.90      0.89       765
weighted avg       0.90      0.89      0.89       765


--- CÁC CHỈ SỐ BỔ SUNG ---
ROC-AUC Score: 96.19%

Ma trận nhầm lẫn (Confusion Matrix):
            Đoán: 0  Đoán: 1
Thực tế: 0      390       73
Thực tế: 1       11      291
